In [2]:
from collections import defaultdict
import graphviz

class DFA:
    def __init__(self, states, alphabet, transitions, start_state, final_states):
        self.states = states
        self.alphabet = alphabet
        self.transitions = transitions
        self.start_state = start_state
        self.final_states = final_states

def minimize_dfa(dfa):
    P0 = [
        [s for s in dfa.states if s in dfa.final_states],
        [s for s in dfa.states if s not in dfa.final_states]
    ]
    P0 = [x for x in P0 if x]
    P1 = []
    
    while P0 != P1:
        P1 = P0.copy()
        P0 = []
        
        for group in P1:
            subgroups = defaultdict(list)
            
            for state in group:
                signature = []
                for symbol in dfa.alphabet:
                    if (state, symbol) in dfa.transitions:
                        dest = dfa.transitions[(state, symbol)]
                        for i, g in enumerate(P1):
                            if dest in g:
                                signature.append(i)
                                break
                    else:
                        signature.append(None)
                
                subgroups[tuple(signature)].append(state)
            
            for key in subgroups:
                subgroups[key].sort()
            
            P0.extend(subgroups.values())
    
    P0.sort(key=lambda x: x[0])
    
    new_states = [tuple(sorted(group)) for group in P0]
    new_transitions = {}
    new_final_states = []
    
    new_start_state = None
    for group in new_states:
        if dfa.start_state in group:
            new_start_state = group
            break
    
    for group in new_states:
        state = group[0]
        if any(s in dfa.final_states for s in group):
            new_final_states.append(group)
            
        for symbol in dfa.alphabet:
            if (state, symbol) in dfa.transitions:
                dest = dfa.transitions[(state, symbol)]
                for dest_group in new_states:
                    if dest in dest_group:
                        new_transitions[(group, symbol)] = dest_group
                        break
    
    return DFA(new_states, dfa.alphabet, new_transitions, new_start_state, new_final_states)

def print_transitions(dfa, title="DFA Transitions"):
    print(f"\n{title}:")
    print("-" * 50)
    print(f"{'State':<20}{'0':<20}{'1':<20}")
    print("-" * 50)
    
    sorted_states = sorted(dfa.states, key=lambda x: x[0] if isinstance(x, tuple) else x)
    
    for state in sorted_states:
        trans_0 = dfa.transitions.get((state, '0'), '-')
        trans_1 = dfa.transitions.get((state, '1'), '-')
        print(f"{str(state):<20}{str(trans_0):<20}{str(trans_1):<20}")
    
    print("\nStart State:", dfa.start_state)
    print("Final States:", dfa.final_states)

def create_transition_diagram(dfa, filename):
    dot = graphviz.Digraph(comment='DFA Transition Diagram')
    dot.attr(rankdir='LR')
    
    # Add initial point
    dot.node('start', '', shape='point')
    
    # Add states
    for state in dfa.states:
        shape = 'doublecircle' if state in dfa.final_states else 'circle'
        dot.node(str(state), str(state), shape=shape)
    
    # Add initial transition
    dot.edge('start', str(dfa.start_state))
    
    # Add transitions
    for (source, symbol), target in dfa.transitions.items():
        dot.edge(str(source), str(target), label=symbol)
    
    # Save the diagram
    dot.render(filename, format='png', cleanup=True)
    print(f"Transition diagram saved as {filename}.png")

def main():
    # Define original DFA with ordered states
    states = {'q0', 'q1', 'q2', 'q3', 'q4'}
    alphabet = {'0', '1'}
    transitions = {
        ('q0', '0'): 'q2', ('q0', '1'): 'q1',
        ('q1', '0'): 'q3', ('q1', '1'): 'q0',
        ('q2', '0'): 'q4', ('q2', '1'): 'q3',
        ('q3', '0'): 'q3', ('q3', '1'): 'q3',
        ('q4', '0'): 'q2', ('q4', '1'): 'q1',
    }
    start_state = 'q0'
    final_states = {'q0', 'q4'}
    
    # Create and process original DFA
    original_dfa = DFA(states, alphabet, transitions, start_state, final_states)
    print_transitions(original_dfa, "Original DFA Transitions")
    create_transition_diagram(original_dfa, "original_dfa")
    
    # Minimize and process minimized DFA
    minimized_dfa = minimize_dfa(original_dfa)
    print_transitions(minimized_dfa, "Minimized DFA Transitions using Equivalence Method")
    create_transition_diagram(minimized_dfa, "minimized_dfa")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'graphviz'